# Entrenamiento Red Neuronal - Detector de Spam en YouTube

**Proyecto I: Introduccion a la IA** - Universitat Politecnica de Valencia - 2026
**Equipo:** Angel del Campo - Samuel Litago - Artur Mas - Pablo

---

## Mejoras respecto al libro Eckroth (2018)

| Aspecto | Libro | Este notebook |
|---|---|---|
| Vectorizacion | Bag-of-Words | TF-IDF con bigramas (5000 features) |
| Features extra | Ninguna | 5 features EDA (URL, mayusculas, longitud...) |
| Capas ocultas | 1 capa ~100 neuronas | 3 capas (256-128-64) |
| Regularizacion | Ninguna | L2 + Early Stopping |
| Optimizador | SGD fijo | Adam |
| Metricas | Solo accuracy | Accuracy + Precision + Recall + F1 + AUC-ROC |

## Paso 1 - Instalar dependencias

In [ ]:
!pip install scikit-learn pandas numpy matplotlib seaborn joblib -q
print('Dependencias instaladas OK')

## Paso 2 - Importaciones

In [ ]:
import os
import re
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    confusion_matrix, classification_report,
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve
)

plt.style.use('dark_background')
print('Importaciones OK')
import sklearn; print('scikit-learn', sklearn.__version__)

## Paso 3 - Subir y cargar el CSV

Ejecuta la celda de subida y selecciona el archivo `Youtube-Spam-Dataset.csv` de tu ordenador.

In [ ]:
# Subir archivo desde tu ordenador a Colab
from google.colab import files
uploaded = files.upload()  # Selecciona Youtube-Spam-Dataset.csv
print('Archivo subido:', list(uploaded.keys()))

In [ ]:
df = pd.read_csv('Youtube-Spam-Dataset.csv')
df = df[['CONTENT', 'CLASS']].rename(columns={'CONTENT': 'text', 'CLASS': 'label'})
df['text']  = df['text'].astype(str).str.strip()
df['label'] = df['label'].astype(int)
df = df[df['text'].str.len() > 2].reset_index(drop=True)

print('Dataset cargado:', len(df), 'comentarios')
print('  Spam   :', df['label'].sum(), '(', round(df['label'].mean()*100,1), '%)')
print('  No Spam:', (df['label']==0).sum(), '(', round((1-df['label'].mean())*100,1), '%)')
df.head()

## Paso 4 - Ingenieria de features (Sprint 2)

Las 5 variables con mayor correlacion con la clase spam, identificadas en el EDA:
- `palabras_spam`: r = 0.69 (predictor mas potente)
- `longitud_chars`: r = 0.34
- `contiene_url`: r = 0.33
- `ratio_mayusculas`: r = 0.05
- `log_exclamaciones`: transformacion log para skewness=37.4

In [ ]:
SPAM_WORDS = re.compile(
    r'subscribe|suscrib|check.?out|my.?channel|free|gratis|click|win|gana|'
    r'giveaway|sorteo|crypto|bitcoin|investment|earn|profit|dm.?me|'
    r'followers|views|likes|watch.?now|promo|discount|descuento|link',
    re.I
)

def engineer_features(texts):
    t = pd.Series(list(texts)).astype(str)
    n_chars   = t.str.len().fillna(0)
    ratio_cap = (t.str.count('[A-Z]') / n_chars.clip(lower=1)).fillna(0)
    has_url   = t.str.contains('https?://|www\\.|bit\\.ly', case=False, na=False).astype(int)
    log_excl  = np.log1p(t.str.count('!').fillna(0))
    spam_wds  = t.apply(lambda x: 1 if SPAM_WORDS.search(x) else 0)
    return np.column_stack([n_chars, ratio_cap, has_url, log_excl, spam_wds]).astype(np.float32)

X_eng = engineer_features(df['text'])
print('Features EDA generadas:', X_eng.shape)
print(pd.DataFrame(X_eng, columns=['longitud','mayusculas','url','exclamaciones','spam_words']).describe().round(2))

## Paso 5 - TF-IDF + division 70/15/15

Mejora vs libro: TF-IDF bigramas en lugar de Bag-of-Words. Division estratificada.

In [ ]:
idx_train, idx_temp = train_test_split(
    df.index, test_size=0.30, random_state=42, stratify=df['label']
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, random_state=42, stratify=df.loc[idx_temp, 'label']
)

print('Train:', len(idx_train), '| Val:', len(idx_val), '| Test:', len(idx_test))

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    strip_accents='unicode',
    min_df=2
)
tfidf.fit(df.loc[idx_train, 'text'])

def build_X(idx):
    X_tf = tfidf.transform(df.loc[idx, 'text']).toarray().astype(np.float32)
    X_e  = engineer_features(df.loc[idx, 'text'].values)
    return np.hstack([X_tf, X_e])

X_train = build_X(idx_train); y_train = df.loc[idx_train, 'label'].values
X_val   = build_X(idx_val);   y_val   = df.loc[idx_val,   'label'].values
X_test  = build_X(idx_test);  y_test  = df.loc[idx_test,  'label'].values

print('Dimension de entrada:', X_train.shape[1], 'features')
print('  TF-IDF: 5000  |  EDA: 5')

## Paso 6 - Arquitectura de la Red Neuronal

```
Input (5005 features)
        |
Dense(256, ReLU) + L2
        |
Dense(128, ReLU) + L2
        |
Dense(64, ReLU)
        |
Dense(1, Sigmoid) -> P(spam)
```

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    batch_size=64,
    learning_rate_init=0.001,
    max_iter=150,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15,
    n_iter_no_change=10,
    verbose=True,
)

print('Arquitectura: 5005 -> 256 -> 128 -> 64 -> 1')
print('Regularizacion: L2 alpha=', mlp.alpha)
print('Early Stopping: patience=', mlp.n_iter_no_change, 'iteraciones')

## Paso 7 - Entrenamiento

In [ ]:
print('Iniciando entrenamiento...')
print('Muestras de entrenamiento:', X_train.shape[0])
print()
mlp.fit(X_train, y_train)
print()
print('Entrenamiento completado en', mlp.n_iter_, 'iteraciones')

## Paso 8 - Curvas de entrenamiento

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
fig.patch.set_facecolor('#0d0d1a')
ax.set_facecolor('#1a1a2e')
ax.grid(True, alpha=0.2, color='#94a3b8')

ax.plot(mlp.loss_curve_, color='#00d4ff', lw=2.5, label='Train loss')
ax.set_title('Curva de Loss - Red Neuronal Spam Detector', color='white', fontsize=13, fontweight='bold')
ax.set_xlabel('Iteraciones', color='#94a3b8')
ax.set_ylabel('Loss', color='#94a3b8')
ax.legend(facecolor='#1a1a2e', labelcolor='white')
ax.tick_params(colors='#94a3b8')

plt.tight_layout()
plt.savefig('curvas_entrenamiento.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print('Guardado: curvas_entrenamiento.png')

## Paso 9 - Evaluacion en el conjunto de test

Mejora vs libro: 5 metricas en lugar de solo accuracy.

In [ ]:
y_prob = mlp.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec  = recall_score(y_test, y_pred)
f1   = f1_score(y_test, y_pred)
auc  = roc_auc_score(y_test, y_prob)

print('=' * 50)
print('   RESULTADOS EN EL CONJUNTO DE TEST')
print('=' * 50)
print('  Accuracy  :', round(acc, 4), '(', round(acc*100, 2), '%)')
print('  Precision :', round(prec, 4))
print('  Recall    :', round(rec, 4))
print('  F1-Score  :', round(f1, 4))
print('  AUC-ROC   :', round(auc, 4))
print('=' * 50)
print()
print(classification_report(y_test, y_pred, target_names=['No Spam', 'Spam']))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0d0d1a')

# Matriz de confusion
ax = axes[0]
ax.set_facecolor('#1a1a2e')
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Spam', 'Spam'],
            yticklabels=['No Spam', 'Spam'],
            ax=ax, linewidths=0.5,
            annot_kws={'size': 18})
ax.set_title('Matriz de Confusion', color='white', fontsize=14, fontweight='bold')
ax.set_xlabel('Prediccion', color='#94a3b8')
ax.set_ylabel('Real', color='#94a3b8')
ax.tick_params(colors='#94a3b8')

# Curva ROC
ax2 = axes[1]
ax2.set_facecolor('#1a1a2e')
fpr, tpr, _ = roc_curve(y_test, y_prob)
ax2.plot(fpr, tpr, color='#00d4ff', lw=2.5, label='AUC = ' + str(round(auc, 3)))
ax2.plot([0, 1], [0, 1], color='#94a3b8', lw=1, ls='--', label='Azar (AUC=0.5)')
ax2.fill_between(fpr, tpr, alpha=0.1, color='#00d4ff')
ax2.set_xlabel('Falsos Positivos', color='#94a3b8')
ax2.set_ylabel('Verdaderos Positivos', color='#94a3b8')
ax2.set_title('Curva ROC', color='white', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right', facecolor='#1a1a2e', labelcolor='white')
ax2.tick_params(colors='#94a3b8')
ax2.grid(True, alpha=0.2, color='#94a3b8')

plt.tight_layout()
plt.savefig('matriz_confusion_roc.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print('Guardado: matriz_confusion_roc.png')

## Paso 10 - Guardar modelo y artefactos

In [ ]:
os.makedirs('model', exist_ok=True)

joblib.dump(mlp,   'model/spam_model.pkl')
print('Modelo guardado : model/spam_model.pkl')

joblib.dump(tfidf, 'model/tfidf_vectorizer.pkl')
print('TF-IDF guardado : model/tfidf_vectorizer.pkl')

metrics = {
    'accuracy' : float(acc),
    'precision': float(prec),
    'recall'   : float(rec),
    'f1'       : float(f1),
    'auc_roc'  : float(auc),
    'model_type'  : 'MLPClassifier - Red Neuronal Superficial',
    'architecture': '5005 -> 256 -> 128 -> 64 -> 1',
    'n_iter'    : int(mlp.n_iter_)
}

with open('model/metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Metricas guardadas: model/metrics.json')

print()
print('=== RESUMEN FINAL ===')
for k, v in metrics.items():
    if isinstance(v, float):
        print(' ', k, ':', round(v, 4))

## Paso 11 - Test de verificacion

In [ ]:
loaded_model = joblib.load('model/spam_model.pkl')
loaded_tfidf = joblib.load('model/tfidf_vectorizer.pkl')

ejemplos = [
    ('Subscribe to my channel! FREE music! Click: http://bit.ly/xxx', 1),
    ('This song is amazing, I listen to it every day', 0),
    ('WIN FREE CRYPTO! Subscribe NOW and earn money! GIVEAWAY!!!!', 1),
    ('I love this band, been a fan since 2008', 0),
]

print('Test con ejemplos conocidos:')
print()
for texto, etiq_real in ejemplos:
    X_tf  = loaded_tfidf.transform([texto]).toarray().astype(np.float32)
    X_e   = engineer_features([texto])
    prob  = float(loaded_model.predict_proba(np.hstack([X_tf, X_e]))[0][1])
    pred  = 'SPAM' if prob >= 0.5 else 'No Spam'
    real  = 'SPAM' if etiq_real == 1 else 'No Spam'
    ok    = 'OK' if (prob >= 0.5) == etiq_real else 'ERROR'
    print('[' + ok + '] ' + pred + ' (' + str(round(prob*100, 1)) + '%)  Real: ' + real)
    print('    "' + texto[:65] + '"')
    print()

## Paso 12 - Descargar artefactos para GitHub

Ejecuta esta celda para descargar los 3 archivos del modelo.
Luego subelos a GitHub en la carpeta `model/`.

In [ ]:
from google.colab import files
print('Descargando artefactos del modelo...')
files.download('model/spam_model.pkl')
files.download('model/tfidf_vectorizer.pkl')
files.download('model/metrics.json')
print()
print('3 archivos descargados. Subelos a GitHub en la carpeta model/')